# Week 37, Day 2: Custom exceptions

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w037d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def load(text):
    try:
        return int(text)
    except ValueError as err:
        raise RuntimeError("couldn't load the setting") from err

try:
    load("ten")
except RuntimeError as err:
    print(err)
    print(type(err.__cause__).__name__)
    print(err.__cause__)

In [ ]:
class ShopError(Exception):
    """Base class for every error the shop raises on purpose."""

class NotFound(ShopError):
    """Something asked for doesn't exist."""

class OutOfStock(ShopError):
    """The item exists but there aren't enough."""

class PaymentFailed(ShopError):
    """The payment service said no."""

class CardDeclined(PaymentFailed):
    """The bank refused the card."""

for cls in [NotFound, CardDeclined]:
    print(cls.__name__, "->", " -> ".join(c.__name__ for c in cls.__mro__[1:-2]))

In [ ]:
class ShopError(Exception):
    pass

class NotFound(ShopError, LookupError):
    pass

class BadQuantity(ShopError, ValueError):
    pass

def parse_quantity(text):
    if not text.isdigit() or int(text) == 0:
        raise BadQuantity(f"quantity must be a whole number above 0, not {text!r}")
    return int(text)

try:
    parse_quantity("-2")
except ValueError as err:          # an older caller that knows nothing of ShopError
    print("ValueError caught:", err)
print(issubclass(NotFound, LookupError), isinstance(NotFound("x"), ShopError))

In [ ]:
class ShopError(Exception):
    pass

class OutOfStock(ShopError):
    def __init__(self, item, wanted, available):
        super().__init__(f"{item}: wanted {wanted}, only {available} left")
        self.item = item
        self.wanted = wanted
        self.available = available

try:
    raise OutOfStock("tea", 5, 2)
except OutOfStock as err:
    print(err)
    print(f"Would you like the {err.available} we have?")

In [ ]:
import json

class ConfigError(Exception):
    pass

def read_config(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError as err:
        raise ConfigError("the settings file isn't valid JSON") from err

read_config('{"theme": "dark",}')

In [ ]:
def attempt(how):
    try:
        {}["missing"]
    except KeyError as err:
        if how == "from err":
            raise ValueError("bad") from err
        if how == "plain":
            raise ValueError("bad")
        raise ValueError("bad") from None

for how in ["from err", "plain", "from None"]:
    try:
        attempt(how)
    except ValueError as e:
        print(f"{how:<10} cause={e.__cause__!r:<16} context={e.__context__!r:<16} hide context={e.__suppress_context__}")

In [ ]:
attempts = []

def charge(amount):
    try:
        if amount > 100:
            raise ValueError("over the limit")
        return "charged"
    except ValueError as err:
        attempts.append(f"failed: {err}")
        raise

try:
    charge(250)
except ValueError as err:
    print("caller saw:", err)
print(attempts)

In [ ]:
def parse_row(row):
    name, age = row.split(",")
    return name, int(age)

def parse_file(lines):
    people = []
    for number, line in enumerate(lines, start=1):
        try:
            people.append(parse_row(line))
        except ValueError as err:
            err.add_note(f"in line {number}: {line!r}")
            raise
    return people

parse_file(["Ada,36", "Bo,40", "Chi,forty"])

In [ ]:
import json

class ShopError(Exception):
    pass

class StorageError(ShopError):
    pass

class JsonOrders:
    def __init__(self, path):
        self.path = path

    def load(self):
        try:
            with open(self.path) as f:
                return json.load(f)
        except FileNotFoundError:
            return []                                   # no file yet: no orders
        except (OSError, json.JSONDecodeError) as err:
            raise StorageError(f"couldn't read orders from {self.path}") from err

with open("orders.json", "w") as f:
    f.write("[1, 2,")
for path in ["none-yet.json", "orders.json"]:
    try:
        print(JsonOrders(path).load())
    except ShopError as err:
        print(f"{err} ({type(err.__cause__).__name__})")

> **Key idea.** Give a program one base exception, with subclasses for the cases callers handle differently, and inherit from a built-in too (`ValueError`, `LookupError`) when the error really is one. Messages are for people (what was wrong, the value, what's expected); attributes are for code. `raise New(...) from err` chains deliberately, setting `__cause__`; a plain raise in `except` sets only `__context__` ("during handling", often an accident); `from None` hides the original. A bare `raise` re-raises unchanged, and `add_note` adds where it happened. Translate low-level errors into your own at each layer boundary, keeping the original as the cause.

### Exercise: A family for a booking system

Create the exceptions for a booking system: `BookingError` (from `Exception`); `NoSuchRoom` (a `BookingError` and a `LookupError`); `RoomTaken` (a `BookingError`) whose `__init__(self, room, by)` stores `room` and `by` and has the message `"room R is taken by B"`; and `BadDates` (a `BookingError` and a `ValueError`).

In [ ]:
class BookingError(Exception):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_family():\n    \"the family\"\n    assert issubclass(NoSuchRoom, BookingError) and issubclass(NoSuchRoom, LookupError)\n    assert issubclass(BadDates, BookingError) and issubclass(BadDates, ValueError)\n    assert issubclass(RoomTaken, BookingError) and not issubclass(RoomTaken, (LookupError, ValueError))\n    assert issubclass(BookingError, Exception) and not issubclass(BookingError, (LookupError, ValueError))\ndef test_taken():\n    \"RoomTaken\"\n    e = RoomTaken(12, \"Ada\")\n    assert str(e) == \"room 12 is taken by Ada\" and e.room == 12 and e.by == \"Ada\"")

### Exercise: Translating a lookup

`ConfigError` is given. Write `get_port(settings)`, which returns `int(settings["port"])`. If the key is missing, raise `ConfigError("no port setting")` chained **from** the `KeyError`; if it isn't a number, raise `ConfigError` with the message `"port must be a number, not 'abc'"` (using the value's `repr`), chained from the `ValueError`. Ports outside 1 to 65535 raise `ConfigError("port out of range: 70000")`, with no cause.

In [ ]:
class ConfigError(Exception):
    pass

def get_port(settings):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ok():\n    \"good ports\"\n    assert get_port({\"port\": \"8080\"}) == 8080 and get_port({\"port\": 1}) == 1\ndef test_errors():\n    \"each error\"\n    cases = [({}, \"no port setting\", KeyError), ({\"port\": \"abc\"}, \"port must be a number, not 'abc'\", ValueError), ({\"port\": \"70000\"}, \"port out of range: 70000\", type(None))]\n    for settings, message, cause in cases:\n        try:\n            get_port(settings)\n            assert False, f\"{settings} should raise ConfigError\"\n        except ConfigError as err:\n            assert str(err) == message, f\"got {str(err)!r}\"\n            assert type(err.__cause__) is cause, f\"{settings}: the cause should be {cause.__name__}, got {err.__cause__!r}\"")

### Exercise: Telling the whole story

Write `explain(err)`, which follows an exception's chain back to the start and returns a list of strings, newest first, each `"TypeName: message"`. Follow `__cause__` if it's set; otherwise follow `__context__`, unless `__suppress_context__` is true. Then add each error's notes, if it has any (`getattr(err, "__notes__", [])`), as `"  note: text"` straight after that error's line.

In [ ]:
def explain(err):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def make(how):\n    try:\n        try:\n            try:\n                int(\"x\")\n            except ValueError as e:\n                e.add_note(\"reading age\")\n                raise\n        except ValueError as e:\n            if how == \"from\":\n                raise KeyError(\"age\") from e\n            if how == \"plain\":\n                raise KeyError(\"age\")\n            raise KeyError(\"age\") from None\n    except KeyError as e:\n        try:\n            raise RuntimeError(\"load failed\") from e\n        except RuntimeError as r:\n            return r\ndef test_chain():\n    \"the chain\"\n    want = [\"RuntimeError: load failed\", \"KeyError: 'age'\", \"ValueError: invalid literal for int() with base 10: 'x'\", \"  note: reading age\"]\n    assert explain(make(\"from\")) == want, f\"got {explain(make('from'))}\"\n    assert explain(make(\"plain\")) == want, \"follow __context__ when there's no cause\"\n    assert explain(make(\"none\")) == want[:2], \"from None hides the context\"\n    assert explain(ValueError(\"alone\")) == [\"ValueError: alone\"]")

### Exercise: Saying where

Write `parse_scores(lines)`. Each line is `"name:score"`; return a dictionary from name to `int` score. If a line fails with a `ValueError` (a missing colon or a bad number), add the note `"line N: 'the line'"` (N counting from 1, the line shown with `repr`) and re-raise the **same** error with a bare `raise`.

In [ ]:
def parse_scores(lines):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ok():\n    \"good lines\"\n    assert parse_scores([\"ada:9\", \"bo:7\"]) == {\"ada\": 9, \"bo\": 7} and parse_scores([]) == {}\ndef test_notes():\n    \"a note on the way up\"\n    for lines, note in [([\"ada:9\", \"bo:seven\"], \"line 2: 'bo:seven'\"), ([\"nocolon\"], \"line 1: 'nocolon'\")]:\n        try:\n            parse_scores(lines)\n            assert False, \"should raise ValueError\"\n        except ValueError as err:\n            assert getattr(err, \"__notes__\", None) == [note], f\"notes: {getattr(err, '__notes__', None)}\"\n            assert err.__cause__ is None and err.__context__ is None, \"re-raise the same error, don't make a new one\"")

### Bug hunt: The empty message

`Overdrawn` should print `balance 20, tried to take 50`, but its `__init__` never passes the message to `Exception`, so `str(err)` is empty. Fix the class.

In [ ]:
class Overdrawn(Exception):
    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        self.message = f"balance {balance}, tried to take {amount}"

try:
    raise Overdrawn(20, 50)
except Overdrawn as err:
    print(err)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the message\"\n    expect_output(\"balance 20, tried to take 50\")\n    e = Overdrawn(1, 2)\n    assert e.balance == 1 and e.amount == 2")

### Bug hunt: The lost cause

`load_user` translates a `KeyError` into a `UserError`, but it throws the original away with `from None`, so the caller can't see which key was missing. Keep the original as the cause, so this prints `no such user` then `'u42'`.

In [ ]:
class UserError(Exception):
    pass

USERS = {"u1": "Ada"}

def load_user(uid):
    try:
        return USERS[uid]
    except KeyError:
        raise UserError("no such user") from None

try:
    load_user("u42")
except UserError as err:
    print(err)
    print(err.__cause__)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"keep the cause\"\n    expect_output(\"no such user\\n'u42'\")")

### Bug hunt: Translated too much

The storage layer should translate only the errors it understands. This one catches `Exception`, so a bug (a `TypeError` from passing a number instead of a string) is disguised as a `StorageError`, and the caller prints `storage problem` when it should crash and show the bug. Catch only `OSError`, so the output is `storage problem` for the missing folder and then `bug: TypeError`.

In [ ]:
class StorageError(Exception):
    pass

def save(path, text):
    try:
        with open(path, "w") as f:
            f.write(text)
    except Exception as err:
        raise StorageError(f"couldn't save to {path}") from err

for path, text in [("no-such-folder/x.txt", "hi"), ("ok.txt", 42)]:
    try:
        save(path, text)
    except StorageError:
        print("storage problem")
    except Exception as err:
        print("bug:", type(err).__name__)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"only what it understands\"\n    expect_output(\"storage problem\\nbug: TypeError\")")

### Puzzle: Translating an error

Arrange the lines into a repository method that turns a missing record into the program's own error, keeping the original as the cause. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
class AppError(Exception):
    pass

class Missing(AppError, LookupError):
    pass

def f(how):
    try:
        [][0]
    except IndexError as err:
        if how == 1:
            raise Missing("gone") from err
        if how == 2:
            raise Missing("gone")
        raise Missing("gone") from None

results = []
for how in [1, 2, 3]:
    try:
        f(how)
    except LookupError as e:
        results.append((type(e.__cause__).__name__, type(e.__context__).__name__, e.__suppress_context__))

p = ___   # results[0]
q = ___   # results[1]
r = ___   # results[2]
s = ___   # str(Missing("gone"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == (\"IndexError\", \"IndexError\", True), \"p\"\n    assert q == (\"NoneType\", \"IndexError\", False), \"q\"\n    assert r == (\"NoneType\", \"IndexError\", True) and s == \"gone\", \"r and s\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a layered loader

Build a small layered loader for member records (one per line, `"id,name,age"`).

- `LoadError(Exception)` is the base. `SourceMissing(LoadError)` is for a missing file. `BadRecord(LoadError, ValueError)` has `__init__(self, line_no, reason)`, stores both, and has the message `"line N: reason"`.
- `read_lines(path)` returns the file's lines (without line endings), or raises `SourceMissing(f"no file {path}")` **from** the `FileNotFoundError`.
- `parse(line_no, line)` returns `(id, name, age)` with `id` and `age` as ints. A line without exactly three fields raises `BadRecord(line_no, "expected 3 fields, got K")`; a bad number raises `BadRecord(line_no, "bad number")` **from** the `ValueError`. A name that is empty after `strip()` raises `BadRecord(line_no, "empty name")`.
- `load(path)` reads and parses every non-blank line (line numbers count every line, from 1) and returns the list of tuples.

In [ ]:
class LoadError(Exception):
    pass

def read_lines(path):
    ___

def parse(line_no, line):
    ___

def load(path):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_classes():\n    \"the family\"\n    assert issubclass(SourceMissing, LoadError) and issubclass(BadRecord, LoadError) and issubclass(BadRecord, ValueError)\n    e = BadRecord(3, \"empty name\")\n    assert str(e) == \"line 3: empty name\" and e.line_no == 3 and e.reason == \"empty name\"\ndef test_parse():\n    \"parsing\"\n    assert parse(1, \"7,Ada,36\") == (7, \"Ada\", 36)\n    for line, reason, cause in [(\"7,Ada\", \"expected 3 fields, got 2\", type(None)), (\"x,Ada,36\", \"bad number\", ValueError), (\"7, ,36\", \"empty name\", type(None))]:\n        try:\n            parse(4, line)\n            assert False, f\"{line!r} should raise\"\n        except BadRecord as err:\n            assert str(err) == f\"line 4: {reason}\", f\"got {str(err)!r}\"\n            assert type(err.__cause__) is cause, f\"{line!r}: cause {err.__cause__!r}\"\ndef test_load():\n    \"loading\"\n    write_file(\"members.txt\", \"1,Ada,36\\n\\n2,Bo,40\\n\")\n    assert load(\"members.txt\") == [(1, \"Ada\", 36), (2, \"Bo\", 40)]\n    write_file(\"bad.txt\", \"1,Ada,36\\n\\n2,Bo\\n\")\n    try:\n        load(\"bad.txt\")\n        assert False\n    except BadRecord as err:\n        assert err.line_no == 3, \"line numbers count blank lines too\"\n    try:\n        load(\"nowhere.txt\")\n        assert False\n    except SourceMissing as err:\n        assert str(err) == \"no file nowhere.txt\" and isinstance(err.__cause__, FileNotFoundError)")

## Warm-up

The `RuntimeError` remembers the `ValueError` that caused it, in `__cause__`. The caller gets an error at the right level ("couldn't load the setting"), and anyone debugging still sees the original detail. That's today's main tool. First, though: what should your own exceptions look like?

## Designing a family

A program, or a package (Week 20), should have **one base exception**, with everything it raises on purpose underneath:

The base class gives callers a choice of how precise to be: `except CardDeclined` for one case, `except PaymentFailed` for a group, `except ShopError` for "anything the shop reported", and bugs (`TypeError`, `AttributeError`) still get through to be found and fixed.

How many classes? Make one for each thing a **caller would handle differently**, and no more. If every caller treats "name too long" and "name has bad characters" the same way, one `InvalidName` with a good message is enough. Python's own `OSError` is a good model: one class, a handful of subclasses (`FileNotFoundError`, `PermissionError`, ...) for the cases people actually handle separately.

**Should it also be a built-in?** If your error really is a kind of `ValueError` or `LookupError`, inherit from both your base and the built-in. Then code that already catches the built-in still works:

Choose `LookupError` rather than `KeyError` for "not found": a `KeyError` subclass shows its message in quotes (`str(KeyError("book 7"))` is `'book 7'`), which looks odd in a message for people.

## Errors that carry details

A message is for people; **attributes** are for code. When a caller might want to act on the details (show the missing item, retry after a delay), keep them on the exception, as in Week 30:

Good messages say **what** was wrong, **with which value**, and if possible **what would be right**: `quantity must be a whole number above 0, not '-2'` beats `invalid input`. Use `!r` so empty strings and stray spaces show up.

## Chaining with raise from

When you catch one error and raise another, write `raise NewError(...) from err`. The new error's `__cause__` is the old one, and a traceback shows both:

The traceback reads "The above exception was the direct cause of the following exception". There are three ways to raise inside an `except`:

| You write | `__cause__` | `__context__` | The traceback shows |
|---|---|---|---|
| `raise New(...) from err` | `err` | `err` | both, "direct cause" |
| `raise New(...)` | `None` | `err` | both, "**During handling** of the above exception, another exception occurred" |
| `raise New(...) from None` | `None` | `err` (hidden) | only the new error |

- `from err` says "this was deliberate: the new error **means** the old one". Use it whenever you translate an error.
- A plain `raise` inside `except` suggests an **accident**: something went wrong while handling the first error. Seeing "During handling of..." in a traceback is often a clue to a bug in an `except` block.
- `from None` hides the original, as Week 25's `spend` did for damaged JSON. Use it only when the original adds nothing for whoever reads the traceback.

## Re-raising and notes

Sometimes you catch an error only to do something (record it, tidy up) and then let it carry on. A **bare `raise`** re-raises the error being handled, traceback intact:

Write `raise`, not `raise err`: both work, but the bare form says "the same error, carrying on" and keeps the traceback cleanest.

Since Python 3.11, `err.add_note(text)` attaches **extra context** without changing the error's type or message. Notes show at the end of the traceback, and live in `err.__notes__`:

Notes are good when the error's **type** is right but it lacks the **where**: a line number, a file name, the record being processed. The caller can still `except ValueError`.

## Translating at the boundary

Week 36 built programs in layers: interface, services, domain, storage. Errors should be translated at the edges between them. The storage layer knows about files and `OSError`s; the services above shouldn't have to. So the storage layer catches the low-level error and raises one of the program's own, chained with `from`:

- Errors it can **handle** (no file yet), it handles.
- Errors it can't handle are **translated**: the service layer and the interface catch `ShopError` and never import `json`. Swap the storage for a database and they don't change.
- The original is kept as `__cause__`, so the detail isn't lost for debugging.
- At the very top, the interface turns `ShopError`s into friendly messages and exit codes (Week 21), and lets bugs crash with a full traceback, because a bug hidden behind "something went wrong" never gets fixed.

## Practice

## Quiz

## Daily challenge

## Before you go

Look at the errors a program of yours raises. Is there one base class? Do the messages say which value was wrong? Where does a low-level error leak up into code that shouldn't know about it? Tomorrow: EAFP and LBYL in depth, and why "check first" can go wrong even when it looks safer.